# FL-DermaBench: benchmarking FedAvg and FedProx against centralized training on DermaMNIST

This notebook reproduces every experiment in the paper **from scratch**, for seeds **42, 43 and 44**.

| Step | What it trains | Runs | Time on a T4 GPU |
|---|---|---|---|
| 1 | Centralized ResNet-18 benchmark (MedMNIST v2 protocol) | 3 (one per seed) | about 30 min |
| 2 | Federated study on **IID** hospitals: FedAvg and FedProx at 0/40/80% stragglers | 18 (6 per seed) | about 4 h |
| 3 | Federated study on **non-IID** hospitals: same 6 cases | 18 (6 per seed) | about 4 h |
| 4 | *Optional:* training-recipe ablation (seed 42 only) | 4 | about 50 min |
| 5 | Results: mean ± std over seeds, FedProx vs FedAvg, figures | – | 1 min |

**Before you start**
- **Colab:** Runtime → Change runtime type → **T4 GPU**, then run the cells from top to bottom.
  The notebook mounts Google Drive and keeps everything in `MyDrive/FL-DermaBench/`
  (dataset, checkpoints, result files and figures). DermaMNIST (about 20 MB) downloads automatically.
- **Local (Mac with Apple GPU, or Linux with CUDA):** `pip install -r requirements.txt`; files go to `./FL-DermaBench/`.

**Interruptions are fine.** Every run saves a checkpoint after every epoch (centralized) or round (federated).
If Colab disconnects, run all cells again: finished runs are reloaded instead of retrained, and the interrupted run
continues from its last saved round. To retrain a run from zero, delete its folder in `checkpoints_derma/`.

**Running on two Colab accounts at once:** Steps 2 and 3 are independent. If both accounts use the same Drive folder
(share `FL-DermaBench` and add a shortcut to it in the second account's `MyDrive`), one account can run Step 2 while the
other runs Step 3. Each account must first run the setup cells (everything above Step 1's training cell).

In [ ]:
# ==== 1. Environment: Colab or local Mac ====
import os, sys, json, time, random, glob, subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/FL-DermaBench"
else:
    PROJECT_DIR = os.path.abspath("./FL-DermaBench")
CKPT_DIR = os.path.join(PROJECT_DIR, "checkpoints_derma")
DATA_DIR = os.path.join(PROJECT_DIR, "data_derma")
FIG_DIR = os.path.join(PROJECT_DIR, "figures_derma")
for d in (CKPT_DIR, DATA_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
NUM_WORKERS = 2 if IN_COLAB else 0
print(f"Colab: {IN_COLAB} | device: {DEVICE} | checkpoints: {CKPT_DIR}")

In [ ]:
# ==== 2. Configuration (benchmark values from the MedMNIST v2 paper) ====
CFG = {
    "seed": 42,
    "batch_size": 128,
    "epochs": 100,
    "lr": 1e-3,
    "milestones": [50, 75],      # learning rate x 0.1 after these epochs (or rounds)
    # federated learning (uses the 28 x 28 ResNet-18)
    "fl_size": 28,
    "num_clients": 4,
    "rounds": 100,               # 100 rounds x 1 local epoch = same training budget as the benchmark
    "local_epochs": 1,
    "split": "iid",              # "iid" or "noniid"
    "dirichlet_alpha": 0.5,
    "min_client_size": 300,
    "local_test_frac": 0.15,     # each hospital keeps 15% of its data to measure per-hospital accuracy
    "algo": "fedavg",            # "fedavg" or "fedprox"
    "mu": 0.01,
    # training recipe (all False/0 = the exact MedMNIST v2 benchmark protocol)
    "augment": False,            # random flips, rotation, crop and slight colour change on each hospital
    "pretrained": False,         # start from public ImageNet ResNet-18 weights
    "label_smoothing": 0.0,      # softer targets against over-confidence
    "weight_decay": 0.0,         # AdamW weight decay against overfitting
    # stragglers: share of hospitals that finish only part of their local training each round
    "stragglers": 0.0,           # 0.0, 0.4 or 0.8 (with 4 hospitals: 0, 2 or 3 stragglers per round)
    # class balancing inside each hospital's training: on for IID hospitals only (set per run), never used centrally
    "balance": True,
}
TARGET_ACC = 0.735 + 0.02        # goal: at least 2 points above the published 0.735
CLASS_NAMES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
PUBLISHED = {28: {"acc": 0.735, "auc": 0.917}, 224: {"acc": 0.754, "auc": 0.920}}

In [ ]:
# ==== 3. Checkpoint helpers ====
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def rng_state():
    st = {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state()}
    if torch.cuda.is_available():
        st["cuda"] = torch.cuda.get_rng_state_all()
    return st

def set_rng_state(st):
    random.setstate(st["python"]); np.random.set_state(st["numpy"]); torch.set_rng_state(st["torch"])
    if "cuda" in st and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(st["cuda"])

def save_ckpt(obj, path):
    tmp = path + ".tmp"            # write then rename: a disconnect never corrupts the last good file
    torch.save(obj, tmp)
    os.replace(tmp, path)

def load_ckpt(path):
    return torch.load(path, map_location="cpu", weights_only=False) if os.path.exists(path) else None

RECIPE_DEFAULTS = {"augment": False, "pretrained": False, "label_smoothing": 0.0, "weight_decay": 0.0,
                   "stragglers": 0.0, "balance": False}

def check_config(saved, keys):
    # runs saved before the recipe switches existed count as the plain benchmark recipe
    diff = {k: (saved.get(k, RECIPE_DEFAULTS.get(k)), CFG[k]) for k in keys
            if saved.get(k, RECIPE_DEFAULTS.get(k)) != CFG[k]}
    if diff:
        raise ValueError(f"Checkpoint was made with different settings {diff}. "
                         "Restore those settings, or delete the run folder to start over.")

def save_json(obj, path):
    with open(path + ".tmp", "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(path + ".tmp", path)

def cpu_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

In [ ]:
# ==== 4. Data: DermaMNIST (official train / val / test split) ====
try:
    import medmnist
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "medmnist"], check=True)
    import medmnist
from medmnist import DermaMNIST
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler

need_download = not os.path.exists(os.path.join(DATA_DIR, "dermamnist.npz"))
raw = {s: DermaMNIST(split=s, download=need_download and s == "train", root=DATA_DIR) for s in ("train", "val", "test")}

class NpDataset(Dataset):
    def __init__(self, imgs, labels, transform):
        self.imgs, self.labels, self.transform = imgs, labels, transform
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        return self.transform(Image.fromarray(self.imgs[i])), int(self.labels[i])

LABELS = {s: raw[s].labels.reshape(-1).astype(int) for s in raw}
NUM_CLASSES = len(CLASS_NAMES)

# Same preprocessing as the benchmark: normalise to [-1, 1]; the 224 version enlarges the 28-pixel image
def make_tf(train):
    # benchmark normalisation, or ImageNet normalisation when starting from ImageNet weights
    mean, std = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) if CFG["pretrained"] else ([.5] * 3, [.5] * 3)
    t = []
    if train and CFG["augment"]:   # skin lesions have no fixed orientation, so flips and rotations are safe
        t += [transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(20),
              transforms.RandomCrop(28, padding=2, padding_mode="reflect"), transforms.ColorJitter(0.1, 0.1, 0.1)]
    return transforms.Compose(t + [transforms.ToTensor(), transforms.Normalize(mean, std)])

TF = {28:  transforms.Compose([transforms.ToTensor(), transforms.Normalize([.5], [.5])]),
      224: transforms.Compose([transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.NEAREST),
                               transforms.ToTensor(), transforms.Normalize([.5], [.5])])}

def get_ds(split, size=28, train=False):
    return NpDataset(raw[split].imgs, LABELS[split], make_tf(train))

BALANCE_CAP = 10   # a very rare image is drawn at most about 10 times per epoch (augmentation makes each copy differ)

def balanced_sampler(labels):
    # Every class present at this hospital gets an equal share of each epoch's draws, except classes so rare
    # that an equal share would repeat their images more than BALANCE_CAP times; those are capped and the
    # remaining share is split equally among the other classes. Classes the hospital does not have stay absent.
    labels = np.asarray(labels); n = len(labels)
    counts = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    present = counts > 0
    mass, fixed = np.zeros(NUM_CLASSES), np.zeros(NUM_CLASSES, dtype=bool)
    for _ in range(NUM_CLASSES):
        free = present & ~fixed
        mass[free] = (1.0 - mass[fixed].sum()) / free.sum()
        over = free & (n * mass / np.maximum(counts, 1) > BALANCE_CAP)
        if not over.any():
            break
        mass[over] = BALANCE_CAP * counts[over] / n
        fixed |= over
    weights = mass[labels] / counts[labels]
    return WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), num_samples=n, replacement=True)

def make_loader(ds, indices=None, shuffle=False, balance_labels=None):
    if indices is not None:
        ds = Subset(ds, list(indices))
    sampler = balanced_sampler(balance_labels) if balance_labels is not None else None
    return DataLoader(ds, batch_size=CFG["batch_size"], shuffle=shuffle and sampler is None, sampler=sampler,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

print({s: len(LABELS[s]) for s in LABELS})
for s in ("train", "val", "test"):
    counts = np.bincount(LABELS[s], minlength=NUM_CLASSES)
    print(f"{s:5s}", dict(zip(CLASS_NAMES, counts.tolist())))
majority = np.bincount(LABELS["train"]).argmax()
print(f"\nAlways predicting '{CLASS_NAMES[majority]}' would score "
      f"{(LABELS['test'] == majority).mean():.1%} test accuracy: the bar any real model must beat.")

In [ ]:
# ==== 4b. Example image per class ====
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(2.2 * NUM_CLASSES, 2.6))
for c, ax in enumerate(axes):
    i = int(np.where(LABELS["train"] == c)[0][0])
    ax.imshow(raw["train"].imgs[i]); ax.set_title(CLASS_NAMES[c]); ax.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "class_examples.png"), dpi=150); plt.show()

In [ ]:
# ==== 5. Model and evaluation ====
from torchvision import models
from sklearn.metrics import roc_auc_score, f1_score

def build_model(size):
    if CFG["pretrained"]:      # public ImageNet weights: every hospital starts from the same model
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    else:
        m = models.resnet18(weights=None, num_classes=NUM_CLASSES)  # from scratch, as in the benchmark
    if size == 28:   # benchmark's 28-pixel ResNet-18: 3x3 first convolution, no max-pooling
        m.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        m.maxpool = nn.Identity()
    return m

@torch.no_grad()
def predict(model, loader):
    model.eval()
    ys, probs = [], []
    for x, y in loader:
        probs.append(F.softmax(model(x.to(DEVICE)), dim=1).cpu()); ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(probs).numpy()

def metrics(y, p):
    pred = p.argmax(1)
    present = np.unique(y)
    auc = roc_auc_score(y, p, multi_class="ovr", average="macro", labels=list(range(NUM_CLASSES))) \
        if len(present) == NUM_CLASSES else float("nan")      # AUC needs every class in the set
    recalls = [float((pred[y == c] == c).mean()) for c in present]
    mel = CLASS_NAMES.index("mel")
    return {"acc": float((pred == y).mean()), "auc": float(auc),
            "macro_f1": float(f1_score(y, pred, average="macro", labels=list(range(NUM_CLASSES)), zero_division=0)),
            "bal_acc": float(np.mean(recalls)),                      # average recall over classes
            "mel_recall": float((pred[y == mel] == mel).mean()) if (y == mel).any() else float("nan")}

def evaluate(model, loader):
    return metrics(*predict(model, loader))

## Step 1: centralized benchmark, seeds 42, 43, 44 (saves after every epoch)

The reference model follows the MedMNIST v2 protocol exactly: trained from scratch without augmentation, Adam lr 0.001,
batch 128, 100 epochs, learning rate × 0.1 after epochs 50 and 75, best epoch chosen by validation AUC.
Published result for ResNet-18 (28) on DermaMNIST: accuracy 0.735, AUC 0.917.

A seed sets the model's starting weights, the order of the training batches and the augmentation draws; for the
federated runs it also sets the hospital split and the straggler schedule. The dataset, test set and recipe are the same
for every seed.

In [ ]:
# ==== 6. Centralized ResNet-18 training (resumable) ====
RECIPE_KEYS = ["augment", "pretrained", "label_smoothing", "weight_decay", "balance"]
CENTRAL_KEYS = ["seed", "batch_size", "lr", "milestones"] + [k for k in RECIPE_KEYS if k != "balance"]

def recipe_tag(include_balance=True):
    parts = [n for n, k in (("aug", "augment"), ("pre", "pretrained")) if CFG[k]]
    if CFG["label_smoothing"] > 0 or CFG["weight_decay"] > 0:
        parts.append("reg")
    if include_balance and CFG["balance"]:
        parts.append("bal")
    return "_" + "_".join(parts) if parts else ""

def make_optimizer(model, lr):
    if CFG["weight_decay"] > 0:
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG["weight_decay"])
    return torch.optim.Adam(model.parameters(), lr=lr)

def loss_fn(logits, y):
    return F.cross_entropy(logits, y, label_smoothing=CFG["label_smoothing"])

def run_central(size):
    name = f"central_r18_{size}{recipe_tag(include_balance=False)}_s{CFG['seed']}"   # centralized: never balanced
    run_dir = os.path.join(CKPT_DIR, name); os.makedirs(run_dir, exist_ok=True)
    last_path, best_path = os.path.join(run_dir, "last.pt"), os.path.join(run_dir, "best.pt")
    print(f"\n=== {name} ===")

    set_seed(CFG["seed"])
    model = build_model(size).to(DEVICE)
    opt = make_optimizer(model, CFG["lr"])
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=CFG["milestones"], gamma=0.1)
    start, best_auc, history = 0, -1.0, []

    ck = load_ckpt(last_path)
    if ck:
        check_config(ck["config"], CENTRAL_KEYS)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["optimizer"]); sched.load_state_dict(ck["scheduler"])
        start, best_auc, history = ck["epoch"], ck["best_auc"], ck["history"]
        set_rng_state(ck["rng"])
        if start < CFG["epochs"]:
            print(f"Resuming from epoch {start + 1}")

    train_loader = make_loader(get_ds("train", size, train=True), shuffle=True)
    val_loader = make_loader(get_ds("val", size))
    for epoch in range(start, CFG["epochs"]):
        t0 = time.time(); model.train(); loss_sum, seen = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = loss_fn(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item() * len(y); seen += len(y)
        sched.step()
        val = evaluate(model, val_loader)
        history.append({"epoch": epoch + 1, "train_loss": loss_sum / seen,
                        **{f"val_{k}": v for k, v in val.items()}, "time_s": round(time.time() - t0, 1)})
        if val["auc"] > best_auc:                       # benchmark rule: keep the best validation AUC
            best_auc = val["auc"]
            save_ckpt({"model": cpu_state(model), "epoch": epoch + 1, "val": val}, best_path)
        save_ckpt({"model": cpu_state(model), "optimizer": opt.state_dict(), "scheduler": sched.state_dict(),
                   "epoch": epoch + 1, "best_auc": best_auc, "history": history, "rng": rng_state(),
                   "config": dict(CFG)}, last_path)
        save_json(history, os.path.join(run_dir, "history.json"))
        h = history[-1]
        if (epoch + 1) % 10 == 0 or epoch + 1 == CFG["epochs"]:      # saved every epoch, printed every 10
            print(f"  epoch {epoch + 1:3d}/{CFG['epochs']} | loss {h['train_loss']:.3f} | val acc {h['val_acc']:.3f} | "
                  f"val AUC {h['val_auc']:.3f}")

    best = load_ckpt(best_path)
    model.load_state_dict(best["model"])
    test = evaluate(model, make_loader(get_ds("test", size)))
    pub = PUBLISHED[size]
    result = {"run": name, "recipe": recipe_tag(include_balance=False).strip("_") or "benchmark", **test, "best_epoch": best["epoch"],
              "published_acc": pub["acc"], "published_auc": pub["auc"]}
    save_json(result, os.path.join(run_dir, "test.json"))
    verdict = "MATCHES" if abs(test["acc"] - pub["acc"]) <= 0.02 and abs(test["auc"] - pub["auc"]) <= 0.01 else "DIFFERS"
    print(f"\nTest (best epoch {best['epoch']}): ACC {test['acc']:.3f} | AUC {test['auc']:.3f} | macro-F1 {test['macro_f1']:.3f}")
    if recipe_tag(include_balance=False):   # improved recipe: check the accuracy target instead of the reproduction rule
        print(f"Published:              ACC {pub['acc']:.3f}  ->  target ACC >= {TARGET_ACC:.3f}: "
              f"{'REACHED' if test['acc'] >= TARGET_ACC else 'not yet'} ({test['acc'] - pub['acc']:+.3f})")
    else:
        print(f"Published:              ACC {pub['acc']:.3f} | AUC {pub['auc']:.3f}  ->  {verdict} "
              "(rule: within 0.02 ACC and 0.01 AUC)")
    return model, result

In [ ]:
# ==== 7. Seeds and helpers ====
SEEDS = [42, 43, 44]
BENCHMARK_RECIPE = dict(augment=False, pretrained=False, label_smoothing=0.0, weight_decay=0.0)

def run_benchmarks(seeds=SEEDS):
    """Centralized ResNet-18 benchmark (exact MedMNIST v2 protocol) for each seed."""
    for seed in seeds:
        print("\n" + "*" * 100 + f"\n*  SEED {seed}  |  CENTRALIZED BENCHMARK\n" + "*" * 100)
        CFG.update(BENCHMARK_RECIPE); CFG["seed"] = seed
        run_central(28)
    CFG.update(BENCHMARK_RECIPE); CFG["seed"] = SEEDS[0]

print(f"Seeds: {SEEDS}")

In [ ]:
# TRAINING: centralized benchmark for seeds 42, 43, 44 (about 10 minutes per seed, resumable)
run_benchmarks()

In [ ]:
# ==== 8. Confusion matrix and per-class recall (benchmark, seed 42) ====
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm_size = 28
CFG.update(BENCHMARK_RECIPE); CFG["seed"] = 42
cm_model, _ = run_central(cm_size)          # loads the finished run; no retraining
y_test, p_test = predict(cm_model, make_loader(get_ds("test", cm_size)))
cm = confusion_matrix(y_test, p_test.argmax(1), labels=list(range(NUM_CLASSES)))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(cmap="Blues", values_format="d")
plt.title(f"ResNet-18 ({cm_size}), benchmark, seed 42: test confusion matrix")
plt.savefig(os.path.join(FIG_DIR, f"confusion_matrix_{cm_size}.png"), dpi=150, bbox_inches="tight"); plt.show()
recall = cm.diagonal() / cm.sum(1)
print("Per-class recall:", {n: round(float(r), 3) for n, r in zip(CLASS_NAMES, recall)})

## Federated setup (4 simulated hospitals, saves after every round)

**IID hospitals use class balancing; non-IID hospitals do not.** With balancing, each hospital gives every class it has an
equal share of its own training batches (a very rare image is repeated at most about 10 times per epoch). No images move
between hospitals, and a class a hospital does not have stays absent there.
Each hospital keeps 15% of its images aside to measure its own accuracy, so federated training sees about 85% of the
training set.

In [ ]:
# ==== 9. Hospital split (saved, so a resumed run uses the same hospitals) ====
def make_client_split():
    K = CFG["num_clients"]
    path = os.path.join(CKPT_DIR, f"split_{CFG['split']}_K{K}_s{CFG['seed']}.json")
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    rng = np.random.default_rng(CFG["seed"]); y = LABELS["train"]; idx_all = np.arange(len(y))
    for _ in range(200):
        if CFG["split"] == "iid":
            parts = np.array_split(rng.permutation(idx_all), K)
        else:
            parts = [[] for _ in range(K)]
            for c in range(NUM_CLASSES):
                idx_c = rng.permutation(idx_all[y == c])
                cuts = (np.cumsum(rng.dirichlet([CFG["dirichlet_alpha"]] * K)) * len(idx_c)).astype(int)[:-1]
                for k, chunk in enumerate(np.split(idx_c, cuts)):
                    parts[k].extend(chunk.tolist())
        if min(len(p) for p in parts) >= CFG["min_client_size"]:
            break
    clients = []
    for p in parts:
        p = rng.permutation(np.array(p)); n_test = int(len(p) * CFG["local_test_frac"])
        clients.append({"train": p[n_test:].tolist(), "test": p[:n_test].tolist()})
    save_json(clients, path)
    return clients

In [ ]:
# ==== 10. FedAvg / FedProx (resumable, saved after every round) ====
FL_KEYS = RECIPE_KEYS + ["stragglers", "seed", "batch_size", "lr", "milestones", "fl_size", "num_clients", "local_epochs",
           "split", "dirichlet_alpha", "algo", "mu"]

def weighted_average(states, weights):
    total = float(sum(weights)); avg = {}
    for key, ref in states[0].items():
        avg[key] = sum(s[key] * (w / total) for s, w in zip(states, weights)) if ref.is_floating_point() else ref.clone()
    return avg

def round_lr(r):
    return CFG["lr"] * (0.1 ** sum(r >= m for m in CFG["milestones"]))

def n_stragglers():
    return int(round(CFG["stragglers"] * CFG["num_clients"]))

def strag_label():
    return f"{round(CFG['stragglers'] * 100)}% stragglers"

def method_label():
    return "FedAvg" if CFG["algo"] == "fedavg" else f"FedProx (mu = {CFG['mu']:g})"

def run_label():
    split = "IID" if CFG["split"] == "iid" else "non-IID"
    return (f"{method_label()} | {split} hospitals | {strag_label().upper()} "
            f"({n_stragglers()} of {CFG['num_clients']} hospitals per round)")

def fl_run_name():
    s = round(CFG["stragglers"] * 100)
    strag = f"_strag{s}" if s > 0 else ""                                   # 0% runs keep their original names
    mu = f"_mu{CFG['mu']:g}" if CFG["algo"] == "fedprox" and CFG["mu"] != 0.01 else ""   # mu 0.01 keeps its name
    return (f"fl_{CFG['algo']}_{CFG['split']}_K{CFG['num_clients']}_r18_{CFG['fl_size']}{recipe_tag()}"
            f"{mu}{strag}_s{CFG['seed']}")

def run_federated(balance=None):
    # class balancing for IID hospitals only (the optional recipe ablation switches it off explicitly)
    CFG["balance"] = (CFG["split"] == "iid") if balance is None else balance
    K, size = CFG["num_clients"], CFG["fl_size"]
    name = fl_run_name()
    run_dir = os.path.join(CKPT_DIR, name); os.makedirs(run_dir, exist_ok=True)
    last_path, best_path = os.path.join(run_dir, "last.pt"), os.path.join(run_dir, "best.pt")
    print("\n" + "=" * 100 + f"\n{run_label()}\nrun folder: {name}")
    if CFG["stragglers"] > 0:
        print(f"Each round, {n_stragglers()} randomly chosen hospitals finish only 10-90% of their local training. "
              f"{'FedAvg drops their updates.' if CFG['algo'] == 'fedavg' else 'FedProx keeps their partial updates.'}")
    else:
        print("No stragglers: every hospital finishes its local training every round.")
    print("=" * 100)
    clients = make_client_split()
    for k, c in enumerate(clients):
        print(f"hospital {k}: {len(c['train'])} train | class counts "
              f"{np.bincount(LABELS['train'][c['train']], minlength=NUM_CLASSES).tolist()}")

    set_seed(CFG["seed"])
    model = build_model(size)
    state = {"round": 0, "global": cpu_state(model), "history": [], "best_auc": -1.0}
    ck = load_ckpt(last_path)
    if ck:
        keys = [k for k in FL_KEYS if not (k == "mu" and CFG["algo"] == "fedavg")]   # mu does not apply to FedAvg
        check_config(ck["config"], keys); state = ck["state"]; set_rng_state(ck["rng"])
        if state["round"] < CFG["rounds"]:
            print(f"Resuming at round {state['round'] + 1}")
    model = model.to(DEVICE)

    train_ds, eval_ds = get_ds("train", size, train=True), get_ds("train", size)
    val_loader = make_loader(get_ds("val", size))
    client_loaders = [make_loader(train_ds, c["train"], shuffle=True,     # balancing uses only this hospital's own labels
                                  balance_labels=LABELS["train"][c["train"]] if CFG["balance"] else None)
                      for c in clients]
    client_test_loaders = [make_loader(eval_ds, c["test"]) for c in clients]

    while state["round"] < CFG["rounds"]:
        r, t0 = state["round"], time.time()
        global_sd = state["global"]
        global_params = {n: global_sd[n].to(DEVICE) for n, _ in model.named_parameters()}

        # Which hospitals straggle this round, and how far they get. Seeded by round, so FedAvg and FedProx
        # face exactly the same stragglers, and a resumed run repeats the same pattern.
        rr = np.random.default_rng(CFG["seed"] * 1000 + r)
        is_strag = np.zeros(K, dtype=bool)
        if n_stragglers() > 0:
            is_strag[rr.choice(K, n_stragglers(), replace=False)] = True
        full_steps = [len(l) * CFG["local_epochs"] for l in client_loaders]
        fractions = rr.uniform(0.1, 0.9, size=K)                 # how far each straggler gets this round
        steps = [max(1, int(fractions[k] * n)) if is_strag[k] else n for k, n in enumerate(full_steps)]

        new_states, sizes = [], []
        for k in range(K):
            if is_strag[k] and CFG["algo"] == "fedavg":
                continue                                      # FedAvg: a straggler's unfinished work is dropped
            model.load_state_dict(global_sd)
            opt = make_optimizer(model, round_lr(r))
            model.train(); done = 0
            for _ in range(CFG["local_epochs"]):
                for x, y in client_loaders[k]:
                    if done >= steps[k]:
                        break                                 # a straggler stops part-way through
                    x, y = x.to(DEVICE), y.to(DEVICE)
                    loss = loss_fn(model(x), y)
                    if CFG["algo"] == "fedprox":
                        loss = loss + CFG["mu"] / 2 * sum(((p - global_params[n]) ** 2).sum()
                                                          for n, p in model.named_parameters())
                    opt.zero_grad(); loss.backward(); opt.step(); done += 1
            new_states.append(cpu_state(model)); sizes.append(len(clients[k]["train"]))

        if new_states:
            new_global = weighted_average(new_states, sizes)
        else:
            new_global = global_sd                            # every hospital straggled under FedAvg: no update
        model.load_state_dict(new_global)
        val = evaluate(model, val_loader)
        client_acc = [evaluate(model, l)["acc"] for l in client_test_loaders]
        n_strag = int(is_strag.sum())
        rec = {"round": r + 1, **{f"val_{k}": v for k, v in val.items()}, "client_acc": client_acc,
               "client_gap": max(client_acc) - min(client_acc), "stragglers": n_strag,
               "aggregated": len(new_states), "time_s": round(time.time() - t0, 1)}
        state["history"].append(rec); state["global"], state["round"] = new_global, r + 1
        if val["auc"] > state["best_auc"]:
            state["best_auc"] = val["auc"]
            save_ckpt({"model": new_global, "round": r + 1, "val": val}, best_path)
        save_ckpt({"state": state, "rng": rng_state(), "config": dict(CFG)}, last_path)
        save_json(state["history"], os.path.join(run_dir, "history.json"))
        strag_txt = ""
        if CFG["stragglers"] > 0:
            who = ",".join(f"H{k}" for k in range(K) if is_strag[k])
            strag_txt = (f" | stragglers {who} {'dropped' if CFG['algo'] == 'fedavg' else 'partial'}, "
                         f"aggregated {len(new_states)}/{K}")
        print(f"[{method_label()} | {strag_label()}] round {r + 1}/{CFG['rounds']} | val acc {val['acc']:.3f} | "
              f"val AUC {val['auc']:.3f} | hospital gap {rec['client_gap']:.3f}{strag_txt} | {rec['time_s']}s | saved")

    best = load_ckpt(best_path); model.load_state_dict(best["model"])
    test = evaluate(model, make_loader(get_ds("test", size)))
    result = {"run": name, "recipe": recipe_tag().strip("_") or "benchmark", "method": method_label(),
              "split": CFG["split"], "stragglers": strag_label().split()[0],
              **test, "best_round": best["round"], "client_gap_last_round": state["history"][-1]["client_gap"]}
    save_json(result, os.path.join(run_dir, "test.json"))
    print(f"Test [{run_label()}] (best round {best['round']}): ACC {test['acc']:.3f} | AUC {test['auc']:.3f} | "
          f"macro-F1 {test['macro_f1']:.3f}")
    print(f"Target ACC >= {TARGET_ACC:.3f}: {'REACHED' if test['acc'] >= TARGET_ACC else 'not yet'} "
          f"({test['acc'] - 0.735:+.3f} vs published 0.735)")
    return result

## The federated study: stragglers (0%, 40%, 80%), FedAvg vs FedProx

A **straggler** is a hospital that finishes only part of its local training in a round (slow hardware, a dropped
connection). Each round, a fixed number of hospitals is picked at random to straggle: with 4 hospitals, **40% = 2** and
**80% = 3**. Each straggler stops after 10–90% of its local training.
**FedAvg drops** stragglers' updates; **FedProx (mu = 0.1) keeps** their partial updates, held close to the global model.

Fair comparison: in every round both methods face exactly the same stragglers stopping at the same point.
Recipe: IID = augmentation + ImageNet start + class balancing; non-IID = augmentation + ImageNet start + label smoothing
0.1 + weight decay 1e-4. Each split has 6 runs per seed: FedAvg and FedProx at 0%, 40% and 80% stragglers.

In [ ]:
# ==== 11. Straggler study setup ====
STRAGGLER_LEVELS = [0.0, 0.4, 0.8]
FEDPROX_MU = 0.1
METHODS = [("fedavg", 0.01), ("fedprox", FEDPROX_MU)]    # mu is ignored by FedAvg
STUDY_RECIPE = {   # same recipe for every method within a split (balancing: IID only)
    "iid":    dict(augment=True, pretrained=True, label_smoothing=0.0, weight_decay=0.0),
    "noniid": dict(augment=True, pretrained=True, label_smoothing=0.1, weight_decay=1e-4),
}

def straggler_study(split):
    for level in STRAGGLER_LEVELS:
        print("\n" + "#" * 100 + f"\n#  {round(level * 100)}% STRAGGLERS  |  "
              f"{'IID' if split == 'iid' else 'non-IID'} hospitals\n" + "#" * 100)
        for algo, mu in METHODS:
            CFG.update(STUDY_RECIPE[split])
            CFG.update(split=split, algo=algo, mu=mu, stragglers=level)
            run_federated()
    CFG.update(stragglers=0.0, mu=0.01)

In [ ]:
def seed_study(split, seeds=SEEDS):
    """FedAvg and FedProx at 0%, 40% and 80% stragglers for each seed (6 runs per seed)."""
    for seed in seeds:
        print("\n" + "*" * 100 + f"\n*  SEED {seed}  |  {'IID' if split == 'iid' else 'non-IID'} hospitals\n" + "*" * 100)
        CFG["seed"] = seed
        straggler_study(split)
    CFG.update(BENCHMARK_RECIPE); CFG.update(seed=SEEDS[0], stragglers=0.0, mu=0.01)

## Step 2: IID hospitals, seeds 42, 43, 44

In [ ]:
# TRAINING: IID study, 18 runs (6 per seed), about 4 hours on a T4, resumable
seed_study("iid")

## Step 3: non-IID hospitals, seeds 42, 43, 44

Can run at the same time as Step 2 on a second Colab account (see the top of the notebook).

In [ ]:
# TRAINING: non-IID study, 18 runs (6 per seed), about 4 hours on a T4, resumable
seed_study("noniid")

## Step 4 (optional): training-recipe ablation, seed 42

Reproduces the recipe ablation in the paper: FedAvg on IID hospitals, no stragglers and **no class balancing**,
adding one recipe change at a time. Not needed for the main results; skip it to save about 50 minutes.

In [ ]:
# TRAINING (optional): 4 FedAvg runs, about 50 minutes, resumable
import pandas as pd
ABLATION = [
    ("Benchmark recipe (from scratch, no augmentation)", dict(augment=False, pretrained=False, label_smoothing=0.0, weight_decay=0.0)),
    ("+ augmentation",                                   dict(augment=True,  pretrained=False, label_smoothing=0.0, weight_decay=0.0)),
    ("+ ImageNet initialization",                        dict(augment=True,  pretrained=True,  label_smoothing=0.0, weight_decay=0.0)),
    ("+ label smoothing and weight decay",               dict(augment=True,  pretrained=True,  label_smoothing=0.1, weight_decay=1e-4)),
]
abl_rows = []
for label, recipe in ABLATION:
    CFG.update(recipe); CFG.update(seed=42, split="iid", algo="fedavg", mu=0.01, stragglers=0.0)
    run_federated(balance=False)
    with open(os.path.join(CKPT_DIR, fl_run_name(), "test.json")) as f:
        t = json.load(f)
    abl_rows.append({"recipe": label, "accuracy": t["acc"], "auc": t["auc"], "macro_f1": t["macro_f1"]})
CFG.update(BENCHMARK_RECIPE); CFG.update(seed=42, stragglers=0.0, mu=0.01)
abl = pd.DataFrame(abl_rows).round(3)
abl.to_csv(os.path.join(FIG_DIR, "ablation_seed42.csv"), index=False)
print(abl.to_string(index=False))

## Step 5: results over 3 seeds

Run this after Steps 1–3 have finished (it also works with fewer seeds: the `n_seeds` column shows how many
were found, and unfinished runs are listed). Outputs in `MyDrive/FL-DermaBench/figures_derma/`:
- `summary_3seeds.csv`: mean and standard deviation over seeds for every case (centralized = benchmark protocol)
- `summary_3seeds_pretty.csv`: the same as `mean ± std` text, ready for the paper table
- `summary_all_seeds_long.csv`: one row per seed and case
- `fedprox_vs_fedavg.csv`: FedProx minus FedAvg per setting (mean ± std, and in how many seeds FedProx was better)
- `fig_bar_accuracy_3seeds.png`, `fig_bar_auc_3seeds.png`: bars = mean, error bars = std
- `fig_accuracy_grid_3seeds.png`, `fig_auc_grid_3seeds.png`: mean per round, shaded = min–max over seeds

The hospital gap is taken at the **selected (best-validation-AUC) round**, the same model that is tested.

In [ ]:
# ==== 12. Results over all seeds: mean ± std, FedProx vs FedAvg, figures ====
import pandas as pd
import matplotlib.pyplot as plt

def load_json(p):
    with open(p) as f:
        return json.load(f)

def set_case(split, level, algo, mu):
    CFG.update(STUDY_RECIPE[split]); CFG.update(split=split, algo=algo, mu=mu, stragglers=level)
    CFG["balance"] = split == "iid"            # same rule as training: class balancing for IID hospitals only

CASES = [(split, level) for split in ["iid", "noniid"] for level in STRAGGLER_LEVELS]
HOSP = {"iid": "IID", "noniid": "Non-IID"}
METRICS = ["acc", "auc", "macro_f1", "bal_acc", "mel_recall", "hospital_gap"]
NICE = {"acc": "Accuracy", "auc": "AUC", "macro_f1": "Macro-F1", "bal_acc": "Balanced acc.",
        "mel_recall": "Melanoma recall", "hospital_gap": "Hospital gap"}

def gap_at_round(name, rnd):
    h = load_json(os.path.join(CKPT_DIR, name, "history.json"))
    rec = next((r for r in h if r["round"] == rnd), None)
    return rec["client_gap"] if rec else float("nan")

# 1) collect one row per seed and case
long_rows, missing = [], []
for seed in SEEDS:
    CFG["seed"] = seed
    cname = f"central_r18_28_s{seed}"               # benchmark protocol (from scratch, no augmentation)
    p = os.path.join(CKPT_DIR, cname, "test.json")
    if os.path.exists(p):
        t = load_json(p)
        long_rows.append({"seed": seed, "hospitals": "Centralized", "stragglers_pct": "", "method": "Benchmark",
                          **{m: t.get(m, np.nan) for m in METRICS if m != "hospital_gap"},
                          "hospital_gap": np.nan, "best_round": t.get("best_epoch")})
    else:
        missing.append(cname)
    for split, level in CASES:
        for algo, mu in METHODS:
            set_case(split, level, algo, mu)
            name = fl_run_name()
            p = os.path.join(CKPT_DIR, name, "test.json")
            if not os.path.exists(p):
                missing.append(name); continue
            t = load_json(p)
            long_rows.append({"seed": seed, "hospitals": HOSP[split], "stragglers_pct": round(level * 100),
                              "method": "FedAvg" if algo == "fedavg" else "FedProx",
                              **{m: t.get(m, np.nan) for m in METRICS if m != "hospital_gap"},
                              "hospital_gap": gap_at_round(name, t["best_round"]), "best_round": t["best_round"]})
CFG.update(BENCHMARK_RECIPE); CFG.update(seed=42, stragglers=0.0, mu=0.01)
if missing:
    print(f"{len(missing)} runs not finished yet (left out):"); [print("  ", m) for m in missing]

long = pd.DataFrame(long_rows)
long.to_csv(os.path.join(FIG_DIR, "summary_all_seeds_long.csv"), index=False)

# 2) mean and standard deviation over seeds
KEYS = ["hospitals", "stragglers_pct", "method"]
grp = long.groupby(KEYS, sort=False)
agg = grp[METRICS].agg(["mean", "std"])
agg.columns = [f"{m}_{s}" for m, s in agg.columns]
agg.insert(0, "n_seeds", grp.size())
agg = agg.reset_index()
agg.round(4).to_csv(os.path.join(FIG_DIR, "summary_3seeds.csv"), index=False)

def pm(mean, std, n):
    if pd.isna(mean):
        return "–"
    return f"{mean:.3f}" if n < 2 or pd.isna(std) else f"{mean:.3f} ± {std:.3f}"

pretty = agg[KEYS + ["n_seeds"]].copy()
for m in METRICS:
    pretty[NICE[m]] = [pm(a, b, n) for a, b, n in zip(agg[f"{m}_mean"], agg[f"{m}_std"], agg["n_seeds"])]
pretty.to_csv(os.path.join(FIG_DIR, "summary_3seeds_pretty.csv"), index=False)
print("\nTest-set results, mean ± std over seeds (centralized = MedMNIST v2 benchmark protocol)")
try:
    display(pretty)
except NameError:
    print(pretty.to_string(index=False))

# 3) paired comparison: FedProx minus FedAvg for the same seed and setting
fl_long = long[long.method.isin(["FedAvg", "FedProx"])]
comp_rows = []
for (h, s), g in fl_long.groupby(["hospitals", "stragglers_pct"], sort=False):
    piv = g.pivot_table(index="seed", columns="method", values=METRICS)
    row = {"hospitals": h, "stragglers_pct": s}
    for m in METRICS:
        if ("FedAvg" in piv[m]) and ("FedProx" in piv[m]):
            d = (piv[m]["FedProx"] - piv[m]["FedAvg"]).dropna()
            better = (d < 0) if m == "hospital_gap" else (d > 0)      # a smaller gap is better
            row[f"d_{m}"] = pm(d.mean(), d.std(), len(d))
            row[f"fedprox_better_{m}"] = f"{int(better.sum())}/{len(d)}"
    comp_rows.append(row)
comp = pd.DataFrame(comp_rows)
comp.to_csv(os.path.join(FIG_DIR, "fedprox_vs_fedavg.csv"), index=False)
print("\nFedProx minus FedAvg (same seed, same setting): mean ± std, and in how many seeds FedProx was better")
show_cols = [c for c in ["hospitals", "stragglers_pct", "d_acc", "fedprox_better_acc", "d_auc", "fedprox_better_auc",
                          "d_mel_recall", "d_hospital_gap"] if c in comp.columns]
try:
    display(comp[show_cols])
except NameError:
    print(comp[show_cols].to_string(index=False))

# 4) bar charts with error bars (accuracy and AUC)
RED, BLUE = "tab:red", "tab:blue"
cen_rows = agg[agg.hospitals == "Centralized"]
xlabels = [f"{HOSP[s]}\n{round(l * 100)}% stragglers" for s, l in CASES]
x = np.arange(len(CASES)); w = 0.38
for metric, published, fname in [("acc", 0.735, "fig_bar_accuracy_3seeds.png"), ("auc", 0.917, "fig_bar_auc_3seeds.png")]:
    fig, ax = plt.subplots(figsize=(10, 4.6)); tops = []
    for off, meth, col, lab in [(-w / 2, "FedAvg", RED, "FedAvg"), (w / 2, "FedProx", BLUE, f"FedProx (mu = {FEDPROX_MU:g})")]:
        means, stds = [], []
        for s, l in CASES:
            r = agg[(agg.hospitals == HOSP[s]) & (agg.stragglers_pct == round(l * 100)) & (agg.method == meth)]
            means.append(r[f"{metric}_mean"].iloc[0] if len(r) else np.nan)
            stds.append(r[f"{metric}_std"].iloc[0] if len(r) and not pd.isna(r[f"{metric}_std"].iloc[0]) else 0.0)
        ax.bar(x + off, means, w, yerr=stds, capsize=3, color=col, label=lab)
        for xi, v, e in zip(x + off, means, stds):
            if not np.isnan(v):
                ax.text(xi, v + e + 0.004, f"{v:.3f}", ha="center", fontsize=8); tops.append(v + e)
    if len(cen_rows):
        c = cen_rows[f"{metric}_mean"].iloc[0]
        ax.axhline(c, ls="--", color="k", label=f"Centralized benchmark ({c:.3f})"); tops.append(c)
    ax.axhline(published, ls=":", color="gray", label=f"Published benchmark ({published:.3f})")
    lo = 0.5 if metric == "acc" else 0.75
    ax.set_ylim(lo, max(tops + [published]) + 0.05)
    ax.set_xticks(x); ax.set_xticklabels(xlabels)
    ax.set_ylabel(f"Test {'accuracy' if metric == 'acc' else 'AUC'} (mean ± std, axis starts at {lo:.2f})")
    ax.grid(axis="y", alpha=0.3); ax.legend(frameon=False, fontsize=8, ncol=2)
    ax.set_title(f"Final test {'accuracy' if metric == 'acc' else 'AUC'} per case over {len(SEEDS)} seeds")
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, fname), dpi=200); plt.show()

# 5) per-round grids: mean over seeds, shaded band = min-max over seeds
for key, ylabel, fname in [("val_acc", "Validation accuracy", "fig_accuracy_grid_3seeds.png"),
                           ("val_auc", "Validation AUC", "fig_auc_grid_3seeds.png")]:
    fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True, sharey=True)
    for i, split in enumerate(["iid", "noniid"]):
        for j, level in enumerate(STRAGGLER_LEVELS):
            ax = axes[i, j]
            for (algo, mu), col, lab in zip(METHODS, [RED, BLUE], ["FedAvg", f"FedProx (mu = {FEDPROX_MU:g})"]):
                curves = []
                for seed in SEEDS:
                    CFG["seed"] = seed; set_case(split, level, algo, mu)
                    hp = os.path.join(CKPT_DIR, fl_run_name(), "history.json")
                    if os.path.exists(hp):
                        h = load_json(hp)
                        if len(h) == CFG["rounds"]:
                            curves.append([r[key] for r in h])
                if curves:
                    a = np.array(curves); rounds = np.arange(1, a.shape[1] + 1)
                    ax.plot(rounds, a.mean(0), color=col, label=f"{lab} (n={len(curves)})")
                    if len(curves) > 1:
                        ax.fill_between(rounds, a.min(0), a.max(0), color=col, alpha=0.15)
            ax.set_title(f"{HOSP[split]}, {round(level * 100)}% stragglers"); ax.grid(alpha=0.3)
            if i == 1: ax.set_xlabel("Round")
            if j == 0: ax.set_ylabel(ylabel)
            ax.legend(frameon=False, fontsize=7, loc="lower right")
    CFG.update(BENCHMARK_RECIPE); CFG.update(seed=42, stragglers=0.0, mu=0.01)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, fname), dpi=200); plt.show()

print("\nSaved to", FIG_DIR, ": summary_3seeds.csv, summary_3seeds_pretty.csv, summary_all_seeds_long.csv,",
      "fedprox_vs_fedavg.csv, fig_bar_accuracy_3seeds.png, fig_bar_auc_3seeds.png,",
      "fig_accuracy_grid_3seeds.png, fig_auc_grid_3seeds.png")